In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    df['day'] = pd.to_datetime(df['day'])
    return df

def check(meter, measurement, threshold, direction='min', top=20):
    df = fetch_daily(meter, measurement)
    if direction == 'min':
        anomaly = df[df['min_val'] < threshold]
    else:
        anomaly = df[df['max_val'] > threshold]
    print(f'{meter} {measurement} 이상값 ({len(anomaly)}건):')
    print(anomaly[['day', 'min_val', 'max_val']].head(top).to_string())
    print(f'양수 구간: {(df["max_val"] > 0).any()}, 음수 구간: {(df["min_val"] < 0).any()}')
    print()

In [2]:
# H1.Z17: I2, I3 음수
check('H1.Z17', 'I2', -10)
check('H1.Z17', 'I3', -10)

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z17 I2 이상값 (81건):
          day     min_val    max_val
0  2018-01-01  -55.487250 -53.073570
1  2018-01-02  -55.119573 -52.605885
2  2018-01-03 -142.847333 -52.911385
3  2018-01-04  -90.988827 -56.230286
4  2018-01-05  -79.748860 -54.747084
5  2018-01-06  -66.699836 -54.428940
6  2018-01-07  -58.759705 -57.043721
7  2018-01-08 -139.596426 -57.897179
8  2018-01-09  -97.436634 -66.492104
9  2018-01-10  -87.384171 -55.068551
10 2018-01-11  -76.686116 -54.419736
11 2018-01-12 -133.231924 -56.196632
12 2018-01-13  -77.094673 -56.326969
13 2018-01-14  -58.060438 -55.313146
14 2018-01-15 -122.761914 -55.602768
15 2018-01-16 -119.978203 -86.954907
16 2018-01-17 -127.474392 -74.572179
17 2018-01-18 -142.115006 -91.437281
18 2018-01-19 -124.064550 -80.345439
19 2018-01-20 -137.355125 -89.381634
양수 구간: True, 음수 구간: True



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z17 I3 이상값 (79건):
          day    min_val    max_val
0  2018-01-01 -13.774627 -11.136475
1  2018-01-02 -15.873600 -11.467106
2  2018-01-03 -18.748925  -1.155588
3  2018-01-04 -14.568894  -1.084635
4  2018-01-05 -15.356887  -6.156480
5  2018-01-06 -14.604727  -8.290617
6  2018-01-07 -11.768084  -8.063374
7  2018-01-08 -25.910909  -8.534533
8  2018-01-09 -16.600252  -8.065350
9  2018-01-10 -17.814846  -5.166230
10 2018-01-11 -14.080128  -5.944086
11 2018-01-12 -20.496581  -4.162126
12 2018-01-13 -11.359963  -4.876466
13 2018-01-14 -10.060636  -5.760935
14 2018-01-15 -33.621992  -7.809000
15 2018-01-16 -42.922736 -13.246836
16 2018-01-17 -29.639081 -15.399151
17 2018-01-18 -27.153225 -16.493823
18 2018-01-19 -27.453080 -15.831170
19 2018-01-20 -20.019973  -7.768855
양수 구간: True, 음수 구간: True



In [3]:
# H1.Z19: P2 큰 음수, W 음수
check('H1.Z19', 'P2', -100)
check('H1.Z19', 'W', 0)

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z19 P2 이상값 (7건):
          day      min_val      max_val
4  2018-01-05 -3804.587608    -2.911333
7  2018-01-08 -5746.708102    -3.426667
15 2018-01-16 -5416.881240    -2.130167
16 2018-01-17  -598.826667    -3.642128
22 2018-01-23 -5416.881240    -2.130167
23 2018-01-24  -598.826667    -3.642128
29 2018-01-30 -5416.881240  2441.240167
양수 구간: True, 음수 구간: True



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z19 W 이상값 (9건):
          day  min_val  max_val
60 2018-03-02  -161.20  1211.45
61 2018-03-03  -141.14  -139.82
62 2018-03-04  -139.76  -138.43
63 2018-03-05  -138.37  -137.02
64 2018-03-06  -136.96  -135.64
65 2018-03-07  -135.58  -134.25
66 2018-03-08  -134.19  -132.87
67 2018-03-09  -132.82   -62.88
68 2018-03-10   -53.05   177.03
양수 구간: True, 음수 구간: True



In [4]:
# H1.Z24: U1, U3 비정상적으로 낮은 전압
check('H1.Z24', 'U1', 215, direction='min')
check('H1.Z24', 'U3', 215, direction='min')

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z24 U1 이상값 (1건):
            day     min_val     max_val
1527 2022-03-08  207.875833  232.668333
양수 구간: True, 음수 구간: False



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z24 U3 이상값 (1건):
            day     min_val     max_val
1527 2022-03-08  195.879167  233.340833
양수 구간: True, 음수 구간: False



In [5]:
# H1.Z25: f 낮은 주파수
check('H1.Z25', 'f', 49.0)

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z25 f 이상값 (1건):
           day    min_val    max_val
990 2020-09-17  48.096732  50.116917
양수 구간: True, 음수 구간: False



In [6]:
# H1.Z26: I3, P 음수
check('H1.Z26', 'I3', -10)
check('H1.Z26', 'P', -5000)

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z26 I3 이상값 (58건):
          day    min_val   max_val
7  2018-01-08 -43.678755  0.000000
8  2018-01-09 -45.325331  0.000000
9  2018-01-10 -43.083232  0.000000
10 2018-01-11 -44.307328 -0.015276
11 2018-01-12 -45.434923 -0.503663
16 2018-01-17 -42.816304  0.000000
17 2018-01-18 -43.406871 -0.317953
18 2018-01-19 -41.892429  0.000000
21 2018-01-22 -41.431509  0.000000
22 2018-01-23 -46.231640 -0.824619
23 2018-01-24 -39.837927 -0.653595
24 2018-01-25 -46.439873 -0.375346
28 2018-01-29 -39.555156 -0.330567
29 2018-01-30 -37.362675 -0.247671
30 2018-01-31 -24.452734  0.000000
31 2018-02-01 -40.629392  0.000000
32 2018-02-02 -40.711577 -0.480527
35 2018-02-05 -16.471731 -0.391273
37 2018-02-07 -43.724451 -0.350867
38 2018-02-08 -43.035207 -0.309966
양수 구간: True, 음수 구간: True



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z26 P 이상값 (3건):
            day       min_val      max_val
1351 2021-09-13 -18955.558866  7402.994116
1358 2021-09-20 -18109.736461  4658.450986
1675 2022-08-03 -13953.789570  6633.278143
양수 구간: True, 음수 구간: True



In [7]:
# H1.Z28: I1~I3, P 음수
check('H1.Z28', 'I1', -50)
check('H1.Z28', 'I2', -50)
check('H1.Z28', 'I3', -50)
check('H1.Z28', 'P', -10000)

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z28 I1 이상값 (2149건):
          day    min_val     max_val
0  2018-01-01 -90.417292    4.712808
1  2018-01-02 -89.302251   46.333044
3  2018-01-04 -88.898648  101.537371
4  2018-01-05 -87.767374  142.512024
5  2018-01-06 -83.588983   60.562677
6  2018-01-07 -55.460472   70.962401
7  2018-01-08 -68.684920  110.473202
8  2018-01-09 -67.644175  107.112161
9  2018-01-10 -83.893308  179.589570
10 2018-01-11 -85.370246  161.674221
13 2018-01-14 -68.430788   53.797151
14 2018-01-15 -83.924867   69.106508
15 2018-01-16 -56.861615   66.032077
16 2018-01-17 -61.673232  136.379483
17 2018-01-18 -62.213781   78.432936
18 2018-01-19 -63.183674  124.262036
19 2018-01-20 -54.519263   89.731605
21 2018-01-22 -70.070720   94.818738
23 2018-01-24 -51.223326   77.270442
24 2018-01-25 -52.984140   78.253812
양수 구간: True, 음수 구간: True



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z28 I2 이상값 (2176건):
          day     min_val    max_val
0  2018-01-01  -87.855648 -38.208512
1  2018-01-02 -108.510599 -33.599056
2  2018-01-03 -228.155226 -80.665095
3  2018-01-04 -166.422792 -43.131988
4  2018-01-05 -209.821688 -44.844260
5  2018-01-06 -124.938914 -47.459783
6  2018-01-07 -122.500974 -54.857697
7  2018-01-08 -166.475565 -53.125375
8  2018-01-09 -159.155621 -44.257840
9  2018-01-10 -193.468318 -44.655620
10 2018-01-11 -187.793252 -45.705792
11 2018-01-12 -203.119710 -72.700234
12 2018-01-13 -132.357812 -91.296645
13 2018-01-14 -113.108478 -52.030767
14 2018-01-15 -122.942094 -45.066219
15 2018-01-16 -126.943396 -55.968472
16 2018-01-17 -169.360991 -57.329920
17 2018-01-18 -167.764716 -57.498350
18 2018-01-19 -168.453313 -63.399509
19 2018-01-20 -161.166125 -74.636489
양수 구간: True, 음수 구간: True



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z28 I3 이상값 (1618건):
           day     min_val    max_val
87  2018-03-29  -57.556809 -34.660482
88  2018-03-30  -57.361100 -38.220162
89  2018-03-31  -58.340987 -36.415780
90  2018-04-01  -62.313292 -36.158350
91  2018-04-02  -60.065115 -36.065829
92  2018-04-03  -84.452550 -36.604957
93  2018-04-04  -97.821972 -23.723527
95  2018-04-06  -60.469118 -26.719959
96  2018-04-07  -62.873513 -35.105685
98  2018-04-09  -61.249484 -31.318319
99  2018-04-10  -56.413069 -33.624219
100 2018-04-11  -75.069484 -36.691957
101 2018-04-12  -60.486952 -40.835246
102 2018-04-13  -52.721816 -28.189858
103 2018-04-14  -51.291740 -35.546745
105 2018-04-16  -62.868866 -28.043688
106 2018-04-17 -110.159020 -41.146087
107 2018-04-18  -93.089559 -34.804273
108 2018-04-19 -102.151553 -35.627913
109 2018-04-20 -122.280639 -43.699916
양수 구간: True, 음수 구간: True



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z28 P 이상값 (1231건):
          day       min_val        max_val
0  2018-01-01 -57163.256358    1972.058522
1  2018-01-02 -56762.061266   28359.673014
2  2018-01-03 -16230.647751  108433.596276
3  2018-01-04 -53677.615361   62783.801838
4  2018-01-05 -56178.244165   87769.608888
5  2018-01-06 -53772.611307   36367.491331
6  2018-01-07 -34642.778512   41921.933995
7  2018-01-08 -43334.290702   67736.606867
8  2018-01-09 -40528.136740   69184.039132
9  2018-01-10 -51321.786877  112373.323069
10 2018-01-11 -53407.909211  100546.325599
11 2018-01-12 -24120.648049   87713.308784
13 2018-01-14 -43378.509298   30816.048280
14 2018-01-15 -54325.542685   42951.567586
15 2018-01-16 -33613.892206   40858.594442
16 2018-01-17 -38373.445616   82093.950669
17 2018-01-18 -37181.893063   48409.083872
18 2018-01-19 -38301.794051   77926.932993
19 2018-01-20 -32798.797782   57167.456652
20 2018-01-21 -21243.811752   45420.024593
양수 구간: True, 음수 구간: True



In [8]:
# H1.Z29: W 전체 음수 고착 여부
df = fetch_daily('H1.Z29', 'W')
print(f'H1.Z29 W 전체 min: {df["min_val"].min():.2f}, max: {df["max_val"].max():.2f}')
print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
print(f'전체 음수 날짜 수: {(df["max_val"] < 0).sum()}')

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z29 W 전체 min: -2742657.08, max: -123062.34
양수 구간 존재 여부: False
전체 음수 날짜 수: 2191


In [9]:
# H1.Z310: P, W 전체 음수 여부 (PV 계량기 가능성)
for measurement in ['P', 'W']:
    df = fetch_daily('H1.Z310', measurement)
    print(f'H1.Z310 {measurement} min: {df["min_val"].min():.2f}, max: {df["max_val"].max():.2f}')
    print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
    print(f'데이터 시작일: {df["day"].min().date()}, 종료일: {df["day"].max().date()}')
    print()

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z310 P min: -80127.39, max: 6.86
양수 구간 존재 여부: True
데이터 시작일: 2020-06-26, 종료일: 2023-12-31



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z310 W min: -328192.28, max: -642.32
양수 구간 존재 여부: False
데이터 시작일: 2020-06-26, 종료일: 2023-12-31



In [10]:
# H2.T.Z34: U1~U3, f 값 0 구간 확인 (정전 또는 계량기 오류)
for measurement in ['U1', 'U2', 'U3', 'f']:
    check('H2.T.Z34', measurement, 1)

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H2.T.Z34 U1 이상값 (4건):
           day  min_val     max_val
795 2020-03-06      0.0  230.300000
796 2020-03-07      0.0    0.000000
797 2020-03-08      0.0    0.000000
798 2020-03-09      0.0  233.251667
양수 구간: True, 음수 구간: False



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H2.T.Z34 U2 이상값 (4건):
           day  min_val     max_val
795 2020-03-06      0.0  230.100000
796 2020-03-07      0.0    0.000000
797 2020-03-08      0.0    0.000000
798 2020-03-09      0.0  233.258333
양수 구간: True, 음수 구간: False



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H2.T.Z34 U3 이상값 (4건):
           day  min_val  max_val
795 2020-03-06      0.0    231.0
796 2020-03-07      0.0      0.0
797 2020-03-08      0.0      0.0
798 2020-03-09      0.0    234.1
양수 구간: True, 음수 구간: False



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H2.T.Z34 f 이상값 (4건):
           day  min_val    max_val
795 2020-03-06      0.0  50.060000
796 2020-03-07      0.0   0.000000
797 2020-03-08      0.0   0.000000
798 2020-03-09      0.0  50.115917
양수 구간: True, 음수 구간: False



In [11]:
# H2.ZE66: PF max=11.33 (역률 1 초과 불가)
check('H2.ZE66', 'PF', 1.1, direction='max')

H2.ZE66 PF 이상값 (37건):
           day   min_val   max_val
4   2022-03-28  0.580303  8.099685
46  2022-05-09  0.578208  7.196474
86  2022-06-18  0.587522  6.578965
100 2022-07-02  0.581138  7.225730
104 2022-07-06  0.569088  5.153636
161 2022-09-01  0.475064  3.836029
173 2022-09-13  0.564416  4.281962
175 2022-09-15  0.468867  4.481195
186 2022-09-26  0.574308  9.313368
196 2022-10-06  0.482282  4.119800
202 2022-10-12  0.576269  8.130378
228 2022-11-07  0.560221  2.845919
232 2022-11-11  0.563079  3.229875
284 2023-01-02  0.555310  6.189416
300 2023-01-18  0.554328  8.677680
308 2023-01-26  0.477449  9.070222
324 2023-02-11  0.559101  5.563863
327 2023-02-14  0.553815  4.474093
368 2023-03-27  0.556395  6.136187
392 2023-04-20  0.474091  2.606225
양수 구간: True, 음수 구간: False



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


In [12]:
# H2.ZE67: P3 max=248380 극단값, PF max=8.45
check('H2.ZE67', 'P3', 50000, direction='max')
check('H2.ZE67', 'PF', 1.1, direction='max')

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H2.ZE67 P3 이상값 (1건):
           day       min_val       max_val
639 2023-12-21  9.048482e-08  248380.21133
양수 구간: True, 음수 구간: True

H2.ZE67 PF 이상값 (26건):
           day   min_val   max_val
0   2022-03-22  0.479417  6.242496
27  2022-04-18  0.560515  3.452680
40  2022-05-01  0.556148  3.929957
97  2022-06-27  0.575394  1.118621
122 2022-07-22  0.552209  3.061262
137 2022-08-06  0.549290  6.492708
140 2022-08-09  0.542032  3.631858
153 2022-08-22  0.555637  5.402753
197 2022-10-05  0.555345  1.174470
210 2022-10-18  0.427056  7.846196
277 2022-12-24  0.496818  1.362645
278 2022-12-25  0.490506  5.120502
293 2023-01-09  0.491909  4.568021
295 2023-01-11  0.490075  8.452129
308 2023-01-24  0.481906  3.775566
375 2023-04-01  0.495498  6.798877
389 2023-04-15  0.488398  5.687704
416 2023-05-12  0.510964  6.721913
417 2023-05-13  0.502290  1.164406
599 2023-11-11  0.568266  3.374769
양수 구간: True, 음수 구간: False



In [13]:
# H2.ZE74: U2 min=76.04 (전압 76V는 물리적으로 의심)
check('H2.ZE74', 'U2', 150)

H2.ZE74 U2 이상값 (22건):
          day    min_val    max_val
0  2022-03-18  77.190412  78.001077
1  2022-03-19  77.288900  78.290182
2  2022-03-20  77.429269  78.359048
3  2022-03-21  76.525221  77.832233
4  2022-03-22  76.318821  78.146012
5  2022-03-23  76.320764  77.996016
6  2022-03-24  76.747706  78.262098
7  2022-03-25  76.722692  77.935302
8  2022-03-26  77.240729  78.377271
9  2022-03-27  76.675159  78.412616
10 2022-03-28  76.328311  77.687169
11 2022-03-29  76.040969  78.171218
12 2022-03-30  76.092527  78.012386
13 2022-03-31  76.287349  78.021021
14 2022-04-01  76.172993  77.872962
15 2022-04-02  76.700081  78.219482
16 2022-04-03  76.617133  78.411179
17 2022-04-04  76.675319  77.923853
18 2022-04-05  76.169159  77.917988
19 2022-04-06  76.137857  77.863732
양수 구간: True, 음수 구간: False



/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


In [14]:
# H3.Z312: W 전체 음수 여부
df = fetch_daily('H3.Z312', 'W')
print(f'H3.Z312 W min: {df["min_val"].min():.2f}, max: {df["max_val"].max():.2f}')
print(f'양수 구간 존재 여부: {(df["max_val"] > 0).any()}')
print(f'전체 음수 날짜 수: {(df["max_val"] < 0).sum()}')
print(f'데이터 시작일: {df["day"].min().date()}, 종료일: {df["day"].max().date()}')

/tmp/ipykernel_96443/551144648.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H3.Z312 W min: -586741.53, max: -1487.63
양수 구간 존재 여부: False
전체 음수 날짜 수: 1284
데이터 시작일: 2020-06-26, 종료일: 2023-12-31
